# Phase 2B — Full Fine-Tuning with Focal Loss

**What changes from Phase 2A (Linear Probe + Focal):**

| | Phase 1 / 2A (Linear Probe) | Phase 2B (Full Fine-Tune) |
|---|---|---|
| Backbone weights | Frozen | All trainable |
| Trainable params | ~4K (head only) | ~307M (entire model) |
| Base LR | 1.25e-3 | 5e-5 (25× lower) |
| Per-layer LR | Same for all | Decays toward input (LLRD) |
| Gradient clipping | None | max_norm=1.0 |
| Loss | CrossEntropy (P1) / Focal (P2A) | Focal (kept) |

**Why:** with a frozen backbone, the feature representations are fixed at RETFound-pretrained values. The linear head can only learn a linear separator in that fixed space. For rare classes like R3A (PDR), the pre-trained features may not be linearly separable from R2. Full fine-tuning lets the backbone reshape those representations to be more discriminative for this specific 4-class task.

**Risk:** 307M params trained on a dataset with only 43 R3A patients can overfit. Mitigations: LLRD (early layers barely change), weight decay 0.05, focal loss (concentrates learning on hard examples), early stopping.

**Expected training time:** ~2–10× longer per epoch than LP, depending on GPU.

In [ ]:
import os, sys, json, copy, math, time
from pathlib import Path

# Reduce CUDA memory fragmentation (helps on 12 GB GPUs with large models)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import PIL.ImageFile
PIL.ImageFile.LOAD_TRUNCATED_IMAGES = True

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, accuracy_score, cohen_kappa_score, confusion_matrix

os.chdir(os.path.dirname(os.path.abspath('phase2b_full_finetune.ipynb')))
sys.path.insert(0, '.')
print('Working directory:', os.getcwd())

# ── Shared config (same as Phase 1/2A) ────────────────────────────────────────
N_FOLDS     = 5
MAX_EPOCHS  = 50
PATIENCE    = 10
INPUT_SIZE  = 224
NUM_CLASSES = 4
CLASS_WEIGHTS = [1.0, 1.7851, 9.5294, 15.6774]
SEED        = 42
CLASSES     = ['R0', 'R1', 'R2', 'R3A']

# ── Full fine-tuning specific config ─────────────────────────────────────────
BASE_LR      = 5e-5    # much lower than LP: backbone is sensitive
MIN_LR       = 1e-7
WARMUP_EPOCHS = 5      # ramp up LR before cosine decay
WEIGHT_DECAY = 0.05    # AdamW regularisation
LLRD_DECAY   = 0.75    # per-layer LR multiplier toward input
GRAD_CLIP    = 1.0     # prevents exploding gradients in large ViTs
BATCH_SIZE   = 16      # physical batch per GPU step (reduced for 12 GB GPU)
ACCUM_STEPS  = 2       # gradient accumulation: effective batch = 16 × 2 = 32
                       # gradients are divided by ACCUM_STEPS so training dynamics
                       # are identical to a true batch of 32 images

# ── Focal loss ────────────────────────────────────────────────────────────────
FOCAL_GAMMA = 2.0      # same as Phase 2A

HF_REPO   = 'YukunZhou/RETFound_dinov2_meh'
HF_FILE   = 'RETFound_dinov2_meh.pth'
CV_OUTPUT = Path('output_dir/phase2b_cv')
CV_OUTPUT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cpu':
    print('WARNING: Full fine-tuning on CPU is extremely slow (hours per fold).')
if DEVICE.type == 'cuda':
    props = torch.cuda.get_device_properties(DEVICE)
    print(f'GPU: {props.name}  VRAM: {props.total_memory/1e9:.1f} GB')
print(f'Config: BASE_LR={BASE_LR}, LLRD={LLRD_DECAY}, FOCAL_GAMMA={FOCAL_GAMMA}')
print(f'Batch:  physical={BATCH_SIZE}  accum_steps={ACCUM_STEPS}  effective={BATCH_SIZE*ACCUM_STEPS}')

In [ ]:
class FocalLoss(nn.Module):
    """
    Multiclass focal loss with optional per-class weights.
    Same implementation as Phase 2A — only difference is we feed it into a
    full-fine-tuned model instead of a linear probe.
    """
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        log_p  = F.log_softmax(logits, dim=1)
        log_pt = log_p.gather(1, targets.view(-1, 1)).squeeze(1)
        pt     = log_pt.exp()
        focal_weight = (1.0 - pt) ** self.gamma
        if self.weight is not None:
            alpha = self.weight[targets]
            focal_weight = focal_weight * alpha
            loss = -(focal_weight * log_pt).sum() / alpha.sum()
        else:
            loss = -(focal_weight * log_pt).mean()
        return loss


# Sanity check: gamma=0 focal loss must match PyTorch weighted CE
torch.manual_seed(0)
_w = torch.tensor(CLASS_WEIGHTS)
_fl0 = FocalLoss(gamma=0.0, weight=_w)
_ce  = nn.CrossEntropyLoss(weight=_w)
_logits  = torch.randn(16, 4)
_targets = torch.randint(0, 4, (16,))
_diff = abs(float(_fl0(_logits, _targets)) - float(_ce(_logits, _targets)))
print(f'gamma=0 vs weighted CE diff: {_diff:.2e}  ({"PASS" if _diff < 1e-5 else "FAIL"})')

In [ ]:
# ── Load splits (identical to Phase 1/2A) ─────────────────────────────────────
GRADE = {'R0': 0, 'R1': 1, 'R2': 2, 'R3A': 3}

df_all = pd.read_csv('labels/splits.csv')
df_all['grade_int'] = df_all['retinopathy'].map(GRADE)

df_cv   = df_all[df_all['split'].isin(['train', 'val'])].copy()
df_test = df_all[df_all['split'] == 'test'].copy()

pat_grade = df_cv.groupby('code')['grade_int'].max().reset_index()
pat_grade.columns = ['code', 'strat_grade']
patient_ids   = pat_grade['code'].values
patient_strat = pat_grade['strat_grade'].values

# Same SEED → same folds as Phase 1 and 2A (direct comparison valid)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
fold_assignments = {}
for fold_idx, (_, val_pat_idx) in enumerate(skf.split(patient_ids, patient_strat)):
    for pid in patient_ids[val_pat_idx]:
        fold_assignments[pid] = fold_idx
pat_grade['fold'] = pat_grade['code'].map(fold_assignments)

df_cv = df_cv.reset_index(drop=True)
df_cv['cv_idx'] = df_cv.index

print(f'CV pool : {len(df_cv)} images | {len(patient_ids)} patients')
print(f'Test set: {len(df_test)} images | {df_test["code"].nunique()} patients')
print('Same folds as Phase 1/2A (SEED=42) — direct comparison valid.')

In [ ]:
import argparse
from util.datasets import build_transform

_aug_args = argparse.Namespace(
    input_size=INPUT_SIZE, color_jitter=None,
    aa='rand-m9-mstd0.5-inc1', reprob=0.25, remode='pixel', recount=1,
)
train_transform = build_transform('train', _aug_args)
eval_transform  = build_transform('val',   _aug_args)

class RetinopathyDataset(Dataset):
    def __init__(self, records, transform):
        self.records   = records
        self.transform = transform
    def __len__(self):  return len(self.records)
    def __getitem__(self, idx):
        path, label = self.records[idx]
        return self.transform(Image.open(path).convert('RGB')), label

def make_records(df_subset):
    return [(row.image_path, row.grade_int) for row in df_subset.itertuples()]

print('Dataset helpers ready.')

## Backbone loading — full fine-tune version

Only one line changes versus the linear probe setup:

```python
# Linear probe:   param.requires_grad = ('head' in name)  → only head trainable
# Full fine-tune: param.requires_grad = True               → everything trainable
```

Everything else (checkpoint loading, key remapping, head initialisation) is identical.

In [ ]:
import timm
from huggingface_hub import hf_hub_download
from timm.layers import trunc_normal_

def load_backbone_fft(device, num_classes=NUM_CLASSES, seed=None):
    """Load RETFound-DINOv2 with ALL parameters trainable for full fine-tuning.

    Gradient checkpointing is enabled to reduce activation memory:
    instead of storing all 24 blocks' intermediate activations simultaneously,
    only block inputs are stored and activations are recomputed during backward.
    This cuts activation memory by ~10x at the cost of ~33% more compute.
    """
    if seed is not None:
        torch.manual_seed(seed)
        np.random.seed(seed)
    model = timm.create_model(
        'vit_large_patch14_dinov2.lvd142m',
        pretrained=True, img_size=INPUT_SIZE,
        num_classes=num_classes, drop_path_rate=0.2,
    )
    ckpt_path = hf_hub_download(repo_id=HF_REPO, filename=HF_FILE)
    ckpt      = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    ckpt_model = ckpt['teacher']
    ckpt_model = {k.replace('backbone.', ''): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w12.', 'mlp.fc1.'): v for k, v in ckpt_model.items()}
    ckpt_model = {k.replace('mlp.w3.',  'mlp.fc2.'): v for k, v in ckpt_model.items()}
    state = model.state_dict()
    drop  = [k for k in ckpt_model if k in state and ckpt_model[k].shape != state[k].shape]
    for k in drop:
        print(f'  Skipping mismatched: {k}')
        del ckpt_model[k]
    model.load_state_dict(ckpt_model, strict=False)
    trunc_normal_(model.head.weight, std=2e-5)
    nn.init.zeros_(model.head.bias)
    # Full fine-tune: ALL parameters are trainable
    for param in model.parameters():
        param.requires_grad = True
    # Gradient checkpointing: recompute activations during backward to save memory
    model.set_grad_checkpointing(enable=True)
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)')
    print('Gradient checkpointing: ENABLED')
    return model.to(device)

print('Verifying backbone load...')
_m = load_backbone_fft(DEVICE)
print('OK.')
del _m
torch.cuda.empty_cache()

## Layer-wise learning rate decay (LLRD)

The head gets the full `BASE_LR`. Each layer going toward the input is multiplied by `LLRD_DECAY=0.75`:

```
head           → BASE_LR × 0.75⁰ = 5.00e-5
final norm     → BASE_LR × 0.75¹ = 3.75e-5
blocks[23]     → BASE_LR × 0.75² = 2.81e-5
blocks[22]     → BASE_LR × 0.75³ = 2.11e-5
...
blocks[0]      → BASE_LR × 0.75²⁵ ≈ 2.8e-8  (almost frozen)
patch_embed    → BASE_LR × 0.75²⁶ ≈ 2.1e-8
```

Bias, norm, and positional parameters get no weight decay (standard practice for ViTs).

The cosine warmup schedule then scales the entire LLRD pattern up/down together:
- epoch 0: all LRs × 0.2 (warmup start)
- epoch 4: all LRs × 1.0 (peak, end of warmup)
- epoch 49: all LRs × min_lr/base_lr ≈ 0.002 (cosine minimum)

In [ ]:
def build_llrd_optimizer(model, base_lr, weight_decay, decay=LLRD_DECAY):
    """
    AdamW with layer-wise learning rate decay.
    Each param is matched by name to a depth from the head;
    LR = base_lr × decay^depth.
    Bias, norm, cls_token, pos_embed: no weight decay.
    """
    num_blocks = len(model.blocks)

    def get_depth(name):
        if 'head' in name:
            return 0
        # Final norm (model.norm, not block norms)
        if name.startswith('norm'):
            return 1
        if 'blocks.' in name:
            block_idx = int(name.split('blocks.')[1].split('.')[0])
            # Last block (23) → depth 2; first block (0) → depth 25
            return num_blocks - block_idx + 1
        # patch_embed, cls_token, pos_embed, etc.
        return num_blocks + 2

    def no_decay(name):
        return any(tag in name for tag in ['bias', 'norm', 'cls_token', 'pos_embed'])

    # Group parameters by (depth, no_decay flag)
    groups = {}
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        key = (get_depth(name), no_decay(name))
        groups.setdefault(key, []).append(param)

    param_groups = []
    for (depth, nd), params in sorted(groups.items()):
        lr = base_lr * (decay ** depth)
        param_groups.append({
            'params': params,
            'initial_lr': lr,   # stored so cosine schedule can scale it
            'lr': lr,
            'weight_decay': 0.0 if nd else weight_decay,
        })

    return torch.optim.AdamW(param_groups)


# Print LLRD LR table
print('LLRD learning rate schedule (at peak, epoch 5):')
print(f'  head         :  {BASE_LR * LLRD_DECAY**0:.2e}')
print(f'  final norm   :  {BASE_LR * LLRD_DECAY**1:.2e}')
print(f'  blocks[23]   :  {BASE_LR * LLRD_DECAY**2:.2e}')
print(f'  blocks[20]   :  {BASE_LR * LLRD_DECAY**5:.2e}')
print(f'  blocks[12]   :  {BASE_LR * LLRD_DECAY**13:.2e}')
print(f'  blocks[0]    :  {BASE_LR * LLRD_DECAY**25:.2e}')
print(f'  patch_embed  :  {BASE_LR * LLRD_DECAY**26:.2e}')

In [ ]:
# ── Training helpers ──────────────────────────────────────────────────────────

class EarlyStoppingFFT:
    """
    Early stopping for full fine-tuning.
    Saves the full model state dict to disk (not just the head) because all
    307M parameters change during training.
    """
    def __init__(self, patience, model, checkpoint_path):
        self.patience        = patience
        self.best_auroc      = -float('inf')
        self.counter         = 0
        self.checkpoint_path = Path(checkpoint_path)
        torch.save(model.state_dict(), self.checkpoint_path)

    def step(self, auroc, model):
        if auroc != auroc: auroc = 0.0   # NaN guard
        if auroc > self.best_auroc:
            self.best_auroc = auroc
            self.counter    = 0
            torch.save(model.state_dict(), self.checkpoint_path)
            return False
        self.counter += 1
        return self.counter >= self.patience

    def restore(self, model, device):
        state = torch.load(self.checkpoint_path, map_location=device, weights_only=True)
        model.load_state_dict(state)


def get_lr(epoch, warmup_epochs, max_epochs, base_lr, min_lr):
    """Linear warmup then cosine decay."""
    if epoch < warmup_epochs:
        return base_lr * (epoch + 1) / warmup_epochs
    t = (epoch - warmup_epochs) / max(1, max_epochs - warmup_epochs)
    return min_lr + 0.5 * (base_lr - min_lr) * (1 + math.cos(math.pi * t))


def train_epoch_fft(model, loader, optimizer, criterion, device, scaler, epoch):
    """
    One training epoch for full fine-tuning with gradient accumulation.

    Gradient accumulation:
      ACCUM_STEPS mini-batches of size BATCH_SIZE are processed before taking
      one optimizer step. Loss is divided by ACCUM_STEPS so the effective gradient
      magnitude matches a single batch of size BATCH_SIZE × ACCUM_STEPS.

    Gradient checkpointing (set in load_backbone_fft):
      Activations are recomputed during backward instead of being stored —
      reduces peak activation memory ~10× at ~33% speed cost.

    Gradient clipping:
      Applied after unscaling so the clip threshold is in the original scale.
    """
    model.train()
    head_lr  = get_lr(epoch, WARMUP_EPOCHS, MAX_EPOCHS, BASE_LR, MIN_LR)
    lr_scale = head_lr / BASE_LR
    for pg in optimizer.param_groups:
        pg['lr'] = pg['initial_lr'] * lr_scale

    optimizer.zero_grad()
    total_loss   = 0.0
    n_samples    = 0
    step_count   = 0   # counts how many mini-batches since last optimizer step

    for i, (imgs, labels) in enumerate(loader):
        imgs, labels = imgs.to(device), labels.to(device)
        # Determine if this is the last mini-batch overall
        is_last_batch = (i + 1 == len(loader))
        # Determine if we should take an optimizer step after this mini-batch
        should_step = ((step_count + 1) % ACCUM_STEPS == 0) or is_last_batch

        with torch.cuda.amp.autocast():
            # Divide loss by ACCUM_STEPS so accumulated gradient = true-batch gradient
            loss = criterion(model(imgs), labels) / ACCUM_STEPS

        scaler.scale(loss).backward()
        total_loss += loss.item() * ACCUM_STEPS * len(labels)  # undo /ACCUM_STEPS for logging
        n_samples  += len(labels)
        step_count += 1

        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            step_count = 0

    return total_loss / n_samples, head_lr


@torch.no_grad()
def eval_fold(model, loader, device):
    model.eval()
    all_labels, all_probs = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device)
        with torch.cuda.amp.autocast():
            logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().float()
        all_labels.append(labels)
        all_probs.append(probs)
    return torch.cat(all_labels).numpy(), torch.cat(all_probs).numpy()


def compute_metrics(labels, probs, preds=None):
    if preds is None:
        preds = probs.argmax(axis=1)
    probs_f64 = probs.astype(np.float64)
    probs_f64 = probs_f64 / probs_f64.sum(axis=1, keepdims=True)
    try:
        auroc = roc_auc_score(labels, probs_f64, multi_class='ovr', average='macro',
                              labels=list(range(NUM_CLASSES)))
    except Exception:
        auroc = float('nan')
    acc   = accuracy_score(labels, preds)
    kappa = cohen_kappa_score(labels, preds, weights='quadratic')
    cm    = confusion_matrix(labels, preds, labels=list(range(NUM_CLASSES)))
    sens, spec = [], []
    for i in range(NUM_CLASSES):
        tp = cm[i, i]; fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp; tn = cm.sum() - tp - fn - fp
        sens.append(tp / (tp + fn) if (tp + fn) > 0 else float('nan'))
        spec.append(tn / (tn + fp) if (tn + fp) > 0 else float('nan'))
    return {'auroc': auroc, 'accuracy': acc, 'kappa': kappa,
            'macro_sensitivity': np.nanmean(sens), 'macro_specificity': np.nanmean(spec),
            'sensitivity': np.array(sens), 'specificity': np.array(spec)}


print('Helpers defined.')

## 5-fold CV training loop — full fine-tuning

Differences from Phase 2A:
- `load_backbone_fft` instead of `load_backbone` (all params trainable)
- `build_llrd_optimizer` instead of plain `AdamW` (LLRD per layer)
- `train_epoch_fft` instead of `train_epoch` (LLRD schedule + grad clip)
- `EarlyStoppingFFT` instead of `EarlyStopping` (saves full model to disk)

In [ ]:
weight_tensor = torch.tensor(CLASS_WEIGHTS, dtype=torch.float).to(DEVICE)
criterion_cv  = FocalLoss(gamma=FOCAL_GAMMA, weight=weight_tensor)

oof_labels_all = np.zeros(len(df_cv), dtype=np.int64)
oof_probs_all  = np.zeros((len(df_cv), NUM_CLASSES), dtype=np.float32)
fold_results   = []

for fold in range(N_FOLDS):
    print(f'\n{"="*60}')
    print(f'  FOLD {fold+1}/{N_FOLDS}  [full fine-tune, focal γ={FOCAL_GAMMA}]')
    print(f'{"="*60}')

    val_pats      = pat_grade[pat_grade['fold'] == fold]['code'].values
    train_pats    = pat_grade[pat_grade['fold'] != fold]['code'].values
    df_fold_train = df_cv[df_cv['code'].isin(train_pats)]
    df_fold_val   = df_cv[df_cv['code'].isin(val_pats)]
    val_cv_indices = df_fold_val['cv_idx'].values
    print(f'  Train: {len(df_fold_train)} images | Val: {len(df_fold_val)} images')

    ds_train = RetinopathyDataset(make_records(df_fold_train), train_transform)
    ds_val   = RetinopathyDataset(make_records(df_fold_val),   eval_transform)
    ds_test  = RetinopathyDataset(make_records(df_test),       eval_transform)

    loader_train = DataLoader(ds_train, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=4, pin_memory=True, drop_last=False)
    loader_val   = DataLoader(ds_val,   batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)
    loader_test  = DataLoader(ds_test,  batch_size=BATCH_SIZE, shuffle=False,
                              num_workers=4, pin_memory=True)

    model     = load_backbone_fft(device=DEVICE, seed=SEED + fold)
    optimizer = build_llrd_optimizer(model, BASE_LR, WEIGHT_DECAY, decay=LLRD_DECAY)
    scaler    = torch.cuda.amp.GradScaler()
    ckpt_path = CV_OUTPUT / f'best_fold_{fold}.pth'
    stopper   = EarlyStoppingFFT(patience=PATIENCE, model=model, checkpoint_path=ckpt_path)

    t_start = time.time()
    for epoch in range(MAX_EPOCHS):
        tr_loss, cur_lr = train_epoch_fft(
            model, loader_train, optimizer, criterion_cv, DEVICE, scaler, epoch
        )
        val_labels, val_probs = eval_fold(model, loader_val, DEVICE)
        m = compute_metrics(val_labels, val_probs)
        elapsed = time.time() - t_start
        print(f'  ep {epoch:02d} | lr(head)={cur_lr:.2e} | loss={tr_loss:.4f} | '
              f'AUROC={m["auroc"]:.4f} | sens={m["macro_sensitivity"]:.4f} | '
              f'elapsed={elapsed:.0f}s')
        if stopper.step(m['auroc'], model):
            print(f'  Early stop at epoch {epoch} (best AUROC={stopper.best_auroc:.4f})')
            break

    stopper.restore(model, DEVICE)
    print(f'  Best val AUROC: {stopper.best_auroc:.4f}')

    oof_labels, oof_probs = eval_fold(model, loader_val, DEVICE)
    oof_labels_all[val_cv_indices] = oof_labels
    oof_probs_all[val_cv_indices]  = oof_probs

    test_labels_fold, test_probs_fold = eval_fold(model, loader_test, DEVICE)

    np.save(CV_OUTPUT / f'fold_{fold}_oof_probs.npy',   oof_probs)
    np.save(CV_OUTPUT / f'fold_{fold}_oof_labels.npy',  oof_labels)
    np.save(CV_OUTPUT / f'fold_{fold}_test_probs.npy',  test_probs_fold)
    np.save(CV_OUTPUT / f'fold_{fold}_test_labels.npy', test_labels_fold)

    m_fold = compute_metrics(oof_labels, oof_probs)
    fold_results.append({
        'fold': fold, 'best_auroc': stopper.best_auroc,
        'oof_auroc': m_fold['auroc'], 'oof_kappa': m_fold['kappa'],
        'oof_macro_sens': m_fold['macro_sensitivity'],
        'oof_macro_spec': m_fold['macro_specificity'],
    })
    print(f'  OOF AUROC: {m_fold["auroc"]:.4f}  Kappa: {m_fold["kappa"]:.4f}')

    del model
    torch.cuda.empty_cache()

print('\n' + '='*60)
print('All folds complete.')
np.save(CV_OUTPUT / 'oof_labels_all.npy', oof_labels_all)
np.save(CV_OUTPUT / 'oof_probs_all.npy',  oof_probs_all)
with open(CV_OUTPUT / 'fold_results.json', 'w') as f:
    json.dump(fold_results, f, indent=2)
print(f'Saved to {CV_OUTPUT}/')

## Test ensemble and Youden thresholds

In [ ]:
from sklearn.metrics import roc_curve

# ── Test ensemble: average probabilities across the 5 fold models ─────────────
test_probs_list = [
    np.load(CV_OUTPUT / f'fold_{f}_test_probs.npy').astype(np.float64)
    for f in range(N_FOLDS)
]
test_labels_all = np.load(CV_OUTPUT / 'fold_0_test_labels.npy')
test_probs_ens  = np.mean(test_probs_list, axis=0)
test_probs_ens  = test_probs_ens / test_probs_ens.sum(axis=1, keepdims=True)

np.save(CV_OUTPUT / 'test_ensemble_probs.npy',  test_probs_ens)
np.save(CV_OUTPUT / 'test_ensemble_labels.npy', test_labels_all)
print('Test ensemble saved.')

# ── Youden thresholds on Phase 2B OOF ────────────────────────────────────────
p2b_oof_probs  = np.load(CV_OUTPUT / 'oof_probs_all.npy').astype(np.float64)
p2b_oof_probs  = p2b_oof_probs / p2b_oof_probs.sum(axis=1, keepdims=True)
p2b_oof_labels = np.load(CV_OUTPUT / 'oof_labels_all.npy')

youden_thr_p2b = []
for i in range(NUM_CLASSES):
    fpr, tpr, thrs = roc_curve((p2b_oof_labels == i).astype(int), p2b_oof_probs[:, i])
    j = tpr - fpr
    youden_thr_p2b.append(float(thrs[j.argmax()]))

print(f'Phase 2B Youden thresholds: {[f"{t:.4f}" for t in youden_thr_p2b]}')

def apply_thresholds(probs, thresholds):
    thresholds = np.array(thresholds)
    ratios = probs / thresholds
    above  = probs > thresholds
    return np.where(above.any(axis=1), ratios.argmax(axis=1), probs.argmax(axis=1))

## Results: Phase 1 vs Phase 2A vs Phase 2B

In [ ]:
P1_DIR  = Path('output_dir/phase1_cv')
P2A_DIR = Path('output_dir/phase2a_cv')

def load_oof(d):
    probs  = np.load(d/'oof_probs_all.npy').astype(np.float64)
    labels = np.load(d/'oof_labels_all.npy')
    return labels, probs / probs.sum(axis=1, keepdims=True)

def load_test(d):
    probs  = np.load(d/'test_ensemble_probs.npy').astype(np.float64)
    labels = np.load(d/'test_ensemble_labels.npy')
    return labels, probs / probs.sum(axis=1, keepdims=True)

p1_oof_lbl,  p1_oof_prb  = load_oof(P1_DIR)
p2a_oof_lbl, p2a_oof_prb = load_oof(P2A_DIR)
p2b_oof_lbl, p2b_oof_prb = load_oof(CV_OUTPUT)

p1_tst_lbl,  p1_tst_prb  = load_test(P1_DIR)
p2a_tst_lbl, p2a_tst_prb = load_test(P2A_DIR)
p2b_tst_lbl, p2b_tst_prb = load_test(CV_OUTPUT)

# Load previously saved thresholds
with open('output_dir/phase2c_thresholds/thresholds.json') as f:
    t1 = json.load(f)
with open('output_dir/phase2a_cv/phase2a_summary.json') as f:
    t2a = json.load(f)

youden_p1  = [t1['youden_thresholds'][c] for c in CLASSES]
youden_p2a = [t2a['youden_thresholds_p2a'][c] for c in CLASSES]
youden_p2b = youden_thr_p2b   # computed above from Phase 2B OOF

# Six configurations: 3 models × 2 decision rules
configs = [
    ('P1  CE     Argmax', p1_oof_lbl,  p1_oof_prb,  p1_tst_lbl,  p1_tst_prb,  None),
    ('P1  CE     Youden', p1_oof_lbl,  p1_oof_prb,  p1_tst_lbl,  p1_tst_prb,  youden_p1),
    ('P2A Focal  Argmax', p2a_oof_lbl, p2a_oof_prb, p2a_tst_lbl, p2a_tst_prb, None),
    ('P2A Focal  Youden', p2a_oof_lbl, p2a_oof_prb, p2a_tst_lbl, p2a_tst_prb, youden_p2a),
    ('P2B FFT    Argmax', p2b_oof_lbl, p2b_oof_prb, p2b_tst_lbl, p2b_tst_prb, None),
    ('P2B FFT    Youden', p2b_oof_lbl, p2b_oof_prb, p2b_tst_lbl, p2b_tst_prb, youden_p2b),
]

for split_name, get_lbl, get_prb in [
    ('OOF',  lambda c: (c[1], c[2]),  None),
    ('TEST', lambda c: (c[3], c[4]),  None),
]:
    print()
    print('=' * 100)
    print(f'  {split_name}')
    print('=' * 100)
    hdr = f'{"Configuration":<22} | {"AUROC":>6} | {"Kappa":>6} | {"MacSens":>7} | {"MacSpec":>7} | {" | ".join(f"{c:>6}" for c in CLASSES)}'
    print(hdr)
    print('-' * len(hdr))
    for name, oof_lbl, oof_prb, tst_lbl, tst_prb, thr in configs:
        if split_name == 'OOF':
            lbl, prb = oof_lbl, oof_prb
        else:
            lbl, prb = tst_lbl, tst_prb
        preds = apply_thresholds(prb, thr) if thr is not None else None
        m = compute_metrics(lbl, prb, preds)
        s = m['sensitivity']
        print(f'{name:<22} | {m["auroc"]:>6.4f} | {m["kappa"]:>6.4f} | '
              f'{m["macro_sensitivity"]:>7.4f} | {m["macro_specificity"]:>7.4f} | '
              f'{" | ".join(f"{s[i]:>6.4f}" for i in range(NUM_CLASSES))}')

# ── Save Phase 2B summary ─────────────────────────────────────────────────────
m_p2b_oof = compute_metrics(p2b_oof_lbl, p2b_oof_prb)
m_p2b_tst = compute_metrics(p2b_tst_lbl, p2b_tst_prb)
m_p2b_oof_yo = compute_metrics(p2b_oof_lbl, p2b_oof_prb,
                                apply_thresholds(p2b_oof_prb, youden_p2b))
m_p2b_tst_yo = compute_metrics(p2b_tst_lbl, p2b_tst_prb,
                                apply_thresholds(p2b_tst_prb, youden_p2b))

summary = {
    'focal_gamma': FOCAL_GAMMA,
    'llrd_decay': LLRD_DECAY,
    'base_lr': BASE_LR,
    'youden_thresholds_p2b': {c: youden_p2b[i] for i, c in enumerate(CLASSES)},
    'oof':  {
        'Phase2B+Argmax': {'auroc': m_p2b_oof['auroc'],
                           'macro_sensitivity': m_p2b_oof['macro_sensitivity'],
                           'sensitivity': m_p2b_oof['sensitivity'].tolist()},
        'Phase2B+Youden': {'auroc': m_p2b_oof_yo['auroc'],
                           'macro_sensitivity': m_p2b_oof_yo['macro_sensitivity'],
                           'sensitivity': m_p2b_oof_yo['sensitivity'].tolist()},
    },
    'test': {
        'Phase2B+Argmax': {'auroc': m_p2b_tst['auroc'],
                           'macro_sensitivity': m_p2b_tst['macro_sensitivity'],
                           'sensitivity': m_p2b_tst['sensitivity'].tolist()},
        'Phase2B+Youden': {'auroc': m_p2b_tst_yo['auroc'],
                           'macro_sensitivity': m_p2b_tst_yo['macro_sensitivity'],
                           'sensitivity': m_p2b_tst_yo['sensitivity'].tolist()},
    },
}
with open(CV_OUTPUT / 'phase2b_summary.json', 'w') as f:
    json.dump(summary, f, indent=2, default=float)
print(f'\nSummary saved to {CV_OUTPUT}/phase2b_summary.json')